In [ ]:
#!/usr/bin/env python3

import argparse
import os
import glob
from datetime import datetime
import pyulog
import numpy as np
import pandas as pd

from specify_figure_dirs import tracking_plots_dir, root_dir

RequiredTopics = [
    'tracking_info',
    'vehicle_odometry',  # 'odometry' is typically 'vehicle_odometry' in PX4 logs
    'vehicle_rates_setpoint',
    'vehicle_angular_velocity'  # 'angular_velocity' is typically 'vehicle_angular_velocity'
]


class UlgDataLoader:
    """
    Class to load and process ULG files containing specific topics.
    """
    def __init__(self, file_path=None):
        """
        Initialize the ULG data loader.
        
        Args:
            file_path (str, optional): Path to a specific ULG file. If None, the latest ULG file will be loaded.
        """
        self._required_topics = RequiredTopics
        
        if file_path:
            self._ulg_path = file_path
        else:
            self._ulg_path = self.__find_latest_ulg()
            
        print(f"Loading ULG file: {self._ulg_path}")
        self._ulog_data = None
        self.__dataframes = {}
        self.__time_start = None
        self.__time_end_v1 = None
        self.__time_end_v2 = None
    def __find_latest_ulg(self):
        """
        Find the latest ULG file in the directory structure.
        
        Returns:
            str: Path to the latest ULG file.
        """
        # Assuming the script is run from the project root directory
        ulg_dir = os.path.join(root_dir(), '.artifacts/benchmark/ulgs')
        ulg_files = []
        
        # Recursively find all .ulg files
        for root, _, files in os.walk(ulg_dir):
            for file in files:
                if file.endswith(".ulg"):
                    ulg_files.append(os.path.join(root, file))
        
        if not ulg_files:
            raise FileNotFoundError("No ULG files found in the directory structure.")
        
        # Sort by modification time (newest first)
        ulg_files.sort(key=lambda x: os.path.getmtime(x), reverse=True)
        
        return ulg_files[0]
    
    def load_data(self):
        """
        Load the ULG file and extract the required topics.
        
        Returns:
            dict: Dictionary containing DataFrames for each required topic.
        """
        try:
            self._ulog_data = pyulog.ULog(self._ulg_path)
            
            # Check if all required topics are available
            available_topics = [d.name for d in self._ulog_data.data_list]
            missing_topics = [topic for topic in self._required_topics if topic not in available_topics]
            
            if missing_topics:
                raise ValueError(f"Warning: The following required topics are missing: {missing_topics}")
            

            self.time_range_specification()
            # Extract data for each available required topic
            for topic in self._required_topics:
                if topic in available_topics:
                    data = self._ulog_data.get_dataset(topic)
                    self.__dataframes[topic] = self.__convert_to_dataframe(data)
            
            return self.__dataframes
        
        except Exception as e:
            print(f"Error loading ULG file: {e}")
            return None
    
    def time_range_specification(self):
        """
        Get the time range specification.
        """
        tracking_data = self._ulog_data.get_dataset('tracking_info')
        
        self.__time_start = tracking_data.data['timestamp'][0]
        self.__time_end_v1 = (tracking_data.data['timestamp'][-1] - self.__time_start)/1e6
        
        # get the index of the second row that ref_position[0] == ref_position[0][0]
        ref_position_0 = tracking_data.data['ref_position[0]']
        a_cirle_completed_index = np.where(ref_position_0 == ref_position_0[0])[0][1]
        self.__time_end_v2 = (tracking_data.data['timestamp'][a_cirle_completed_index] - self.__time_start)/1e6
        
    
    
    def __convert_to_dataframe(self, dataset):
        """
        Convert a ULog dataset to a pandas DataFrame.
        
        Args:
            dataset: ULog dataset object.
            
        Returns:
            pd.DataFrame: DataFrame containing the dataset data.
        """
        data_dict = {}
        print(self.__time_start)
        # Convert timestamp to seconds
        timestamp = dataset.data['timestamp'] 
        data_dict['time_from_start'] = (timestamp - self.__time_start) / 1e6  # Convert to seconds from start
        
        # Add all other fields
        for field_name in dataset.data.keys():
            if not (field_name == 'timestamp'):
                data_dict[field_name] = dataset.data[field_name]
        
        df = pd.DataFrame(data_dict)
        
        # time_from_start must greater than 0
        
        # convert all data to numeric
        df = df.loc[lambda df: (df['time_from_start'] >=  0) & (df['time_from_start'] <= self.__time_end_v2), :]
        df = df.apply(pd.to_numeric, errors='raise')
        
        return df
    
    def get_topic_data(self, topic_name):
        """
        Get data for a specific topic.
        
        Args:
            topic_name (str): Name of the topic.
            
        Returns:
            pd.DataFrame: DataFrame containing the topic data.
        """
        if not self.__dataframes:
            self.load_data()
            
        if topic_name in self.__dataframes:
            return self.__dataframes[topic_name]
        else:
            print(f"Topic '{topic_name}' not found in the loaded data.")
            return None
    
    def get_available_topics(self):
        """
        Get a list of available topics in the loaded ULG file.
        
        Returns:
            list: List of available topic names.
        """
        if not self._ulog_data:
            self.load_data()
            
        return [d.name for d in self._ulog_data.data_list]

def main():
    """
    Main function to parse arguments and load ULG data.
    """
    parser = argparse.ArgumentParser(description='Load and process ULG files.')
    parser.add_argument('--file', '-f', type=str, help='Path to a specific ULG file to load.')
    args = parser.parse_args()
    
    # Create ULG data loader
    loader = UlgDataLoader(args.file)
    
    # Load data
    data = loader.load_data()

    if data:
        print("Successfully loaded ULG data.")
        print(f"Available topics: {loader.get_available_topics()}")
        
        # Print a summary of the loaded data
        for topic, df in data.items():
            print(f"\nTopic: {topic}")
            print(f"  Shape: {df.shape}")
            print(f"  Time range: {df['time_from_start'].min():.2f}s to {df['time_from_start'].max():.2f}s")
            print(f"  Columns: {', '.join(df.columns)}")
    else:
        print("Failed to load ULG data.")

if __name__ == "__main__":
    main()#!/usr/bin/env python3

import argparse
import os
import glob
from datetime import datetime
import pyulog
import numpy as np
import pandas as pd

from specify_figure_dirs import tracking_plots_dir, root_dir

RequiredTopics = [
    'tracking_info',
    'vehicle_odometry',  # 'odometry' is typically 'vehicle_odometry' in PX4 logs
    'vehicle_rates_setpoint',
    'vehicle_angular_velocity'  # 'angular_velocity' is typically 'vehicle_angular_velocity'
]


class UlgDataLoader:
    """
    Class to load and process ULG files containing specific topics.
    """
    def __init__(self, file_path=None):
        """
        Initialize the ULG data loader.
        
        Args:
            file_path (str, optional): Path to a specific ULG file. If None, the latest ULG file will be loaded.
        """
        self._required_topics = RequiredTopics
        
        if file_path:
            self._ulg_path = file_path
        else:
            self._ulg_path = self.__find_latest_ulg()
            
        print(f"Loading ULG file: {self._ulg_path}")
        self._ulog_data = None
        self.__dataframes = {}
        self.__time_start = None
        self.__time_end_v1 = None
        self.__time_end_v2 = None
    def __find_latest_ulg(self):
        """
        Find the latest ULG file in the directory structure.
        
        Returns:
            str: Path to the latest ULG file.
        """
        # Assuming the script is run from the project root directory
        ulg_dir = os.path.join(root_dir(), '.artifacts/benchmark/ulgs')
        ulg_files = []
        
        # Recursively find all .ulg files
        for root, _, files in os.walk(ulg_dir):
            for file in files:
                if file.endswith(".ulg"):
                    ulg_files.append(os.path.join(root, file))
        
        if not ulg_files:
            raise FileNotFoundError("No ULG files found in the directory structure.")
        
        # Sort by modification time (newest first)
        ulg_files.sort(key=lambda x: os.path.getmtime(x), reverse=True)
        
        return ulg_files[0]
    
    def load_data(self):
        """
        Load the ULG file and extract the required topics.
        
        Returns:
            dict: Dictionary containing DataFrames for each required topic.
        """
        try:
            self._ulog_data = pyulog.ULog(self._ulg_path)
            
            # Check if all required topics are available
            available_topics = [d.name for d in self._ulog_data.data_list]
            missing_topics = [topic for topic in self._required_topics if topic not in available_topics]
            
            if missing_topics:
                raise ValueError(f"Warning: The following required topics are missing: {missing_topics}")
            

            self.time_range_specification()
            # Extract data for each available required topic
            for topic in self._required_topics:
                if topic in available_topics:
                    data = self._ulog_data.get_dataset(topic)
                    self.__dataframes[topic] = self.__convert_to_dataframe(data)
            
            return self.__dataframes
        
        except Exception as e:
            print(f"Error loading ULG file: {e}")
            return None
    
    def time_range_specification(self):
        """
        Get the time range specification.
        """
        tracking_data = self._ulog_data.get_dataset('tracking_info')
        
        self.__time_start = tracking_data.data['timestamp'][0]
        self.__time_end_v1 = (tracking_data.data['timestamp'][-1] - self.__time_start)/1e6
        
        # get the index of the second row that ref_position[0] == ref_position[0][0]
        ref_position_0 = tracking_data.data['ref_position[0]']
        a_cirle_completed_index = np.where(ref_position_0 == ref_position_0[0])[0][1]
        self.__time_end_v2 = (tracking_data.data['timestamp'][a_cirle_completed_index] - self.__time_start)/1e6
        
    
    
    def __convert_to_dataframe(self, dataset):
        """
        Convert a ULog dataset to a pandas DataFrame.
        
        Args:
            dataset: ULog dataset object.
            
        Returns:
            pd.DataFrame: DataFrame containing the dataset data.
        """
        data_dict = {}
        print(self.__time_start)
        # Convert timestamp to seconds
        timestamp = dataset.data['timestamp'] 
        data_dict['time_from_start'] = (timestamp - self.__time_start) / 1e6  # Convert to seconds from start
        
        # Add all other fields
        for field_name in dataset.data.keys():
            if not (field_name == 'timestamp'):
                data_dict[field_name] = dataset.data[field_name]
        
        df = pd.DataFrame(data_dict)
        
        # time_from_start must greater than 0
        
        # convert all data to numeric
        df = df.loc[lambda df: (df['time_from_start'] >=  0) & (df['time_from_start'] <= self.__time_end_v2), :]
        df = df.apply(pd.to_numeric, errors='raise')
        
        return df
    
    def get_topic_data(self, topic_name):
        """
        Get data for a specific topic.
        
        Args:
            topic_name (str): Name of the topic.
            
        Returns:
            pd.DataFrame: DataFrame containing the topic data.
        """
        if not self.__dataframes:
            self.load_data()
            
        if topic_name in self.__dataframes:
            return self.__dataframes[topic_name]
        else:
            print(f"Topic '{topic_name}' not found in the loaded data.")
            return None
    
    def get_available_topics(self):
        """
        Get a list of available topics in the loaded ULG file.
        
        Returns:
            list: List of available topic names.
        """
        if not self._ulog_data:
            self.load_data()
            
        return [d.name for d in self._ulog_data.data_list]

def main():
    """
    Main function to parse arguments and load ULG data.
    """
    parser = argparse.ArgumentParser(description='Load and process ULG files.')
    parser.add_argument('--file', '-f', type=str, help='Path to a specific ULG file to load.')
    args = parser.parse_args()
    
    # Create ULG data loader
    loader = UlgDataLoader(args.file)
    
    # Load data
    data = loader.load_data()

    if data:
        print("Successfully loaded ULG data.")
        print(f"Available topics: {loader.get_available_topics()}")
        
        # Print a summary of the loaded data
        for topic, df in data.items():
            print(f"\nTopic: {topic}")
            print(f"  Shape: {df.shape}")
            print(f"  Time range: {df['time_from_start'].min():.2f}s to {df['time_from_start'].max():.2f}s")
            print(f"  Columns: {', '.join(df.columns)}")
    else:
        print("Failed to load ULG data.")

if __name__ == "__main__":
    main()

In [4]:
from exp_data_loader import UlgDataLoader
# Create ULG data loader
loader = UlgDataLoader()

# Load data
data = loader.load_data()



data['tracking_info']

Loading ULG file: /home/rec/minco-ros2/artifacts/benchmarks/ulgs/2025-03-10_10-33-07.ulg


,time_from_start,ref_position[0],ref_position[1],ref_position[2],ref_velocity[0],ref_velocity[1],ref_velocity[2],actual_position[0],actual_position[1],actual_position[2],...,ref_vxy,actual_vxy,ref_yb[0],ref_yb[1],ref_yb[2],actual_yb[0],actual_yb[1],actual_yb[2],alpha,cz
835,0.000,-3.174936e-05,-0.003419,6.499994,0.000577,0.088789,0.000166,-0.051223,-0.187023,6.429128,...,0.007884,0.335105,-0.999986,0.005311,0.000704,-0.999987,0.003525,-0.003727,1.834478,0.4
836,0.020,-2.070697e-05,-0.001930,6.499996,0.000515,0.060914,0.000114,-0.045808,-0.180751,6.426545,...,0.003711,0.264685,-0.999978,0.006681,0.000374,-0.999965,0.006128,-0.005666,1.821543,0.4
837,0.036,-1.153391e-05,-0.000948,6.499998,0.000396,0.038098,0.000072,-0.042351,-0.177093,6.424863,...,0.001452,0.224792,-0.999970,0.007791,0.000106,-0.999946,0.007786,-0.006873,1.812651,0.4
838,0.056,-5.045109e-06,-0.000371,6.500000,0.000252,0.020481,0.000039,-0.037393,-0.172342,6.422488,...,0.000420,0.174263,-0.999963,0.008571,-0.000086,-0.999912,0.010162,-0.008566,1.798523,0.4
839,0.076,-1.403269e-06,-0.000094,6.500000,0.000116,0.008205,0.000016,-0.034302,-0.169577,6.420928,...,0.000067,0.146167,-0.999960,0.008979,-0.000175,-0.999886,0.011672,-0.009621,1.788315,0.4
840,0.096,-1.090398e-07,-0.000007,6.500000,0.000023,0.001414,0.000003,-0.029886,-0.166067,6.418745,...,0.000002,0.111189,-0.999959,0.009067,-0.000129,-0.999843,0.013830,-0.011099,1.771452,0.4
841,0.136,0.000000e+00,0.000000,6.500000,0.000000,0.000000,0.000000,-0.024887,-0.166278,6.414134,...,0.000000,0.067970,-1.000000,0.000000,-0.000000,-0.999765,0.017136,-0.013282,1.737664,0.4
842,0.156,1.661923e-07,0.000023,6.500000,0.000026,0.003480,0.000016,-0.023028,-0.166691,6.412466,...,0.000012,0.055297,-1.000000,0.000234,0.000292,-0.999732,0.018372,-0.014056,1.720409,0.4
843,0.176,1.567861e-06,0.000185,6.500001,0.000129,0.013792,0.000062,-0.020386,-0.167588,6.410098,...,0.000190,0.040282,-0.999999,0.001317,0.000690,-0.999683,0.020114,-0.015115,1.685442,0.4
844,0.196,6.090769e-06,0.000619,6.500003,0.000344,0.030745,0.000137,-0.018722,-0.168342,6.408656,...,0.000945,0.032611,-0.999992,0.003739,0.001056,-0.999652,0.021173,-0.015743,1.641588,0.4


In [5]:
df = data['tracking_info']
import numpy as np
df = df.assign(ref_vel_norm=lambda df: np.linalg.norm(df[['ref_velocity[0]', 'ref_velocity[1]', 'ref_velocity[2]']], axis=1))

new_df = df[['ref_vel_norm', 'ref_velocity[0]', 'ref_velocity[1]', 'ref_velocity[2]']]

nnew_df = new_df.assign(ref_vel_x=lambda df:np.abs(df['ref_velocity[0]'])) \
          .assign(ref_vel_y=lambda df:np.abs(df['ref_velocity[1]'])) \
          .assign(ref_vel_z=lambda df:np.abs(df['ref_velocity[2]']))

nnew_df.describe()

,ref_vel_norm,ref_velocity[0],ref_velocity[1],ref_velocity[2],ref_vel_x,ref_vel_y,ref_vel_z
count,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000
mean,0.187462,0.004982,0.187389,0.000770,0.004982,0.187389,0.000770
std,0.214605,0.007083,0.214491,0.000911,0.007083,0.214491,0.000911
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.023049,0.000275,0.023047,0.000064,0.000275,0.023047,0.000064
50%,0.086311,0.000977,0.086305,0.000304,0.000977,0.086305,0.000304
75%,0.305276,0.007337,0.305185,0.001306,0.007337,0.305185,0.001306
max,0.685775,0.023701,0.685359,0.002839,0.023701,0.685359,0.002839
